# Example 6-4: Calculating Velocity Change to Change Inclination Only
### _Fundamentals of Astrodynamics and Applications_, 5th Ed., 2022, pp. 348-349

This notebook demonstrates calculating a simple inclination change.

## Install and Import Libraries
---

First, install `valladopy` if it doesn't already exist in your environment:

In [1]:
!pip install -r ../valladopy_version.txt

Import the relevant `valladopy` modules:

In [2]:
import numpy as np
import valladopy.constants as const
from valladopy.astro.maneuver.transfer import incl_only

## Problem Definition
---

GIVEN:&ensp; $\Delta{i}$ = 15.0°, $v_{initial}$ = 5.892311 km/s (0.745356 ER/TU), $e_{initial}$ = 0.0 ($\phi_{fpa}$ = 0.0)<br>
FIND: &emsp;$\Delta v_{\text{\textit{i only}}}$

In [3]:
delta_i = np.radians(15)  # rad
v_init = 5.892311         # km/s
fpa = 0                   # flight path angle, rad

## Solution
---

Because the orbit is circular, $\phi_{fpa}$ is zero, so we can solve for the change in velocity directly using **Equation 6-19**:

$$
\Delta v_{\text{\textit{i only}}} = 2 \ v_{initial} \cos(\phi_{fpa}) \sin \left( \frac{\Delta i}{2} \right)
$$

which can be done with the `incl_only` routine:

In [4]:
dv_i_only = incl_only(delta_i, v_init, fpa)

print(f'delta-v (incl only): {dv_i_only:.6f} km/s')

delta-v (incl only): 1.538202 km/s


Now suppose we have an elliptical orbit with the following parameters:

In [5]:
ecc = 0.3
p = 17858.7836         # semiparameter, km
argp = np.radians(30)  # argument of periapsis, rad
nu = np.radians(330)   # true anomaly, rad

Find the semimajor axis using **Equation 1-10**:

$$
a = \frac{p}{1 - e^2}
$$

Then, find the position using the trajectory equation (**Equation 1-24**):

$$
r = \frac{p}{1 + e \cos(\nu)}
$$

The velocity's magnitude is calculated from **Equation 1-31**:

$$
v = \sqrt{\frac{2 \mu}{r} - \frac{\mu}{a}}
$$

Finally, the flight path angle is determined from using both **Equation 2-97** and **Equation 2-98**:

$$
\tan(\phi_{fpa}) = \frac{e \sin(\nu)}{1 + e \cos(\nu)}
$$

In [6]:
a = p / (1 - ecc ** 2)
r = p / (1 + ecc * np.cos(nu))
v_init = np.sqrt((2 * const.MU / r) - (const.MU / a))
fpa = np.arctan2(ecc * np.sin(nu), 1 + ecc * np.cos(nu))

print(f'a: \t{a:.3f}\tkm')
print(f'r: \t{r:.3f}\tkm')
print(f'v: \t{v_init:.8f}\tkm/s')
print(f'fpa: \t{np.degrees(fpa):.3f}\t\tdeg')

a: 	19625.037	km
r: 	14175.802	km
v: 	5.99382403	km/s
fpa: 	-6.790		deg


This makes sense because the satellite is heading towards perigee — thus, the negative sign.

The change in velocity can be calculated using the same `dv_i_only` routine:

In [7]:
dv_i_only = incl_only(delta_i, v_init, fpa)

print(f'delta-v (incl only): {dv_i_only:.6f} km/s')

delta-v (incl only): 1.553727 km/s


For elliptical cases, we must check both nodes — the true anomaly for the other node is $\nu - 180^\circ$:

In [8]:
nu_node2 = nu - np.pi
r = p / (1 + ecc * np.cos(nu_node2))
v_init = np.sqrt((2 * const.MU / r) - (const.MU / a))
fpa = np.arctan2(ecc * np.sin(nu_node2), 1 + ecc * np.cos(nu_node2))

print(f'nu (node #2): \t{np.degrees(nu_node2):.0f}\t\tdeg')
print(f'r: \t\t{r:.3f}\tkm')
print(f'v: \t\t{v_init:.8f}\tkm/s')
print(f'fpa: \t\t{np.degrees(fpa):.3f}\t\tdeg')

nu (node #2): 	150		deg
r: 		24127.219	km
v: 		3.56801693	km/s
fpa: 		11.456		deg


The positive value is correct because we're headed towards apogee. The velocity change is now:

In [9]:
dv_i_only = incl_only(delta_i, v_init, fpa)

print(f'delta-v (incl only): {dv_i_only:.6f} km/s')

delta-v (incl only): 0.912883 km/s


Notice the reduced overall velocity — you should check both nodes for elliptical orbits!